# Azure SLM Fine-tuning (QLoRA) — RTX 4080 16GB

Trains a LoRA adapter on top of a small instruct model using Azure docs Q/A pairs.
Config here is tuned for 16GB VRAM specifically — batch size 2, grad accumulation 8
(effective batch 16), max sequence length 768. If you OOM anyway, drop max_length to 512
before touching batch size further.

**Prerequisites:** run `prepare_hf_dataset.py` first, or have `azure_qa_pairs.jsonl` ready locally.


In [ ]:
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset, load_from_disk


## 1. Confirm CUDA is visible before anything else

In [ ]:
assert torch.cuda.is_available(), "CUDA not available — fix your torch/CUDA install before continuing"
print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## 2. Config — tuned for 4080 16GB, not generic defaults

In [ ]:
BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"   # swap for Phi-3-mini or Llama-3.2-3B if you prefer
QA_DATASET_DIR = "./azure_qa_dataset"      # from prepare_hf_dataset.py's save_to_disk
QA_JSONL_FALLBACK = "./azure_qa_pairs.jsonl"  # used if the DatasetDict dir isn't there
OUTPUT_DIR = "./azure-slm-lora"

MAX_LENGTH = 768        # drop to 512 first if you OOM — most QA pairs are shorter than this anyway
BATCH_SIZE = 2          # per-device — don't raise this before raising grad accumulation
GRAD_ACCUM = 8           # effective batch size = BATCH_SIZE * GRAD_ACCUM = 16
NUM_EPOCHS = 3           # don't push past 3-4 on a small dataset — catastrophic forgetting risk
LEARNING_RATE = 2e-4


## 3. Load base model in 4-bit + LoRA config

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb_config, device_map="auto"
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 4. Load data and check the split sizes before you train on them

In [ ]:
import os

if os.path.isdir(QA_DATASET_DIR):
    dataset = load_from_disk(QA_DATASET_DIR)
else:
    print("No prepared DatasetDict found, falling back to raw jsonl (no dedupe/quality filter applied)")
    raw = load_dataset("json", data_files=QA_JSONL_FALLBACK, split="train")
    raw = raw.train_test_split(test_size=0.1, seed=42)
    dataset = {"train": raw["train"], "validation": raw["test"]}

print(dataset)


In [ ]:
def format_example(example):
    prompt = (
        f"<|im_start|>system\nYou are an assistant answering questions about "
        f"Microsoft Azure. Answer only from what you know to be accurate; say "
        f"so if you're unsure.<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['response']}<|im_end|>"
    )
    tokenized = tokenizer(prompt, truncation=True, max_length=MAX_LENGTH, padding="max_length")
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

train_ds = dataset["train"].map(format_example, remove_columns=dataset["train"].column_names)
eval_ds = dataset["validation"].map(format_example, remove_columns=dataset["validation"].column_names)
print(f"train: {len(train_ds)}, eval: {len(eval_ds)}")


## 5. Training args

Watch `nvidia-smi` in a terminal (`watch -n 1 nvidia-smi`) during the first few steps.
If you OOM immediately rather than partway through, it's the batch size / MAX_LENGTH combo — fix that before anything else.


In [ ]:
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    bf16=True,
    fp16=False,
    logging_steps=20,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    report_to="none",
    gradient_checkpointing=True,  # trades compute for memory — needed at this VRAM budget
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
)


## 6. Train

In [ ]:
trainer.train()


## 7. Save the adapter — this is what push_model_to_hf.py merges into the base model

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Adapter saved to {OUTPUT_DIR}")


## 8. Quick manual eval — don't trust the loss curve alone

Loss going down tells you nothing about whether answers are actually correct.
Run a handful of real questions and read the output yourself.


In [ ]:
test_questions = [
    "How do I create an Azure Storage account using the Azure CLI?",
    "What's the difference between Azure Functions consumption plan and premium plan?",
    "How do I set up autoscaling for an Azure App Service?",
]

model.eval()
for q in test_questions:
    prompt = (
        f"<|im_start|>system\nYou are an assistant answering questions about "
        f"Microsoft Azure.<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=250, do_sample=False)
    print("Q:", q)
    print("A:", tokenizer.decode(out[0], skip_special_tokens=True).split("assistant")[-1])
    print("-" * 80)


## Next steps

1. Read the eval outputs above. If they're incoherent or off-topic, don't proceed — go back to
   the QA pair quality or hyperparameters first.
2. Run `push_model_to_hf.py` to merge this adapter into the base model and push to the Hub.
3. Quantize the merged model with AutoAWQ before serving with vLLM — that's a separate step,
   not covered here.
